# lncRNA-CpG-miRNA Network Analysis
## Meth3D-Net V6 DMBs + TCGA PanCancer Atlas (40 GB)

### Data sources:
| File | Content |
|---|---|
| `...merged_HumanMethylation27_HumanMethylation450.betaValue_whitelisted.tsv` | ~490K CpG probes × ~10K TCGA samples |
| `pancanMiRs_EBadjOnProtocolPlatformWithoutRepsWithUnCorrectMiRs_08_04_16.csv` | miRNA expression × ~10K TCGA samples |
| `PanCanAtlas_miRNA_sample_information_list.txt` | Sample annotations (cancer type) |

### Strategy (memory-safe for 40 GB):
- Scan methylation TSV **once** with line-by-line reading — extracts only lncRNA promoter probes
- Load miRNA matrix fully (smaller file)
- Align on TCGA barcodes (first 15 chars)
- Correlate lncRNA promoter methylation vs miRNA abundance across all 33 cancer types

**Outputs:** `lncRNA_candidates.csv`, `lncRNA_miRNA_pairs.csv`, `ceRNA_triplets_validated.csv`, `high_priority_triplets.csv`, `Fig_lncRNA_candidates.pdf/png`


In [1]:
# ═══════════════════════════════════════════════════════════════
# CELL 0 — Setup
# ═══════════════════════════════════════════════════════════════
import os, sys, gzip, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

WORK_DIR = '/kaggle/working'
OUT_DIR  = f'{WORK_DIR}/lncRNA_candidates'
os.makedirs(OUT_DIR, exist_ok=True)

# V6 DMB files — two FLAT directories (no per-chromosome subdirs):
#   DIR_A: chr20-22, chrX, chrY  (top-level)
#   DIR_B: chr1-19               (subfolder with a SPACE in the name)
V6_FLAT_DIRS = [
    '/kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6',
    '/kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6/Methylation Paper cpG_v6',
    f'{WORK_DIR}/V6_results',   # fallback if running after V6 notebook locally
]

# LncRNA matrix directory (the 6 GPL platform files)
LNCRNA_MATRIX_DIR = '/kaggle/input/datasets/neetuaashi/lncrna-matrix/LncRNA_matrix'

CHROMS = [f'chr{i}' for i in list(range(1,23))+['X']]

print(f'Output : {OUT_DIR}')
print(f'LncRNA matrix dir: {LNCRNA_MATRIX_DIR}')
print(f'  Exists: {os.path.exists(LNCRNA_MATRIX_DIR)}')
if os.path.exists(LNCRNA_MATRIX_DIR):
    files = os.listdir(LNCRNA_MATRIX_DIR)
    print(f'  Files: {sorted(files)}')


Output : /kaggle/working/lncRNA_candidates
LncRNA matrix dir: /kaggle/input/datasets/neetuaashi/lncrna-matrix/LncRNA_matrix
  Exists: True
  Files: ['GSE16256-GPL10999_series_matrix.txt', 'GSE16256-GPL11154_series_matrix.txt', 'GSE16256-GPL13393_series_matrix.txt', 'GSE16256-GPL16791_series_matrix.txt', 'GSE16256-GPL9052_series_matrix.txt', 'GSE16256-GPL9115_series_matrix.txt']


In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 0b — Inspect all available input files
# ═══════════════════════════════════════════════════════════════
print('=== /kaggle/input tree ===')
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.count(os.sep) - '/kaggle/input'.count(os.sep)
    if depth > 4:
        dirs.clear()
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root) or "input"}/  [{len(files)} files]')
    for f in sorted(files)[:5]:
        sz = os.path.getsize(os.path.join(root, f)) / 1024
        print(f'{indent}  {f}  ({sz:.0f} KB)')
    if len(files) > 5:
        print(f'{indent}  ... and {len(files)-5} more')


=== /kaggle/input tree ===
input/  [0 files]
  datasets/  [0 files]
    neetuaashi/  [0 files]
      methylation-paper-cpg-v6/  [41 files]
        Fig1_Genome_Accuracy.pdf  (40 KB)
        Fig1_Genome_Accuracy.png  (187 KB)
        Fig3_HLA_Zoom.png  (1066 KB)
        V6_genome_summary.csv  (3 KB)
        chr20_V6_ct_scores.csv  (10104 KB)
        ... and 36 more
        Methylation Paper cpG_v6/  [135 files]
          chr10_V6_ct_scores.csv  (18749 KB)
          chr10_V6_dmb_p.csv  (457 KB)
          chr10_V6_dmb_q.csv  (996 KB)
          chr10_V6_metrics.json  (1 KB)
          chr10_V6_model_best.pt  (495 KB)
          ... and 130 more
      methylation-data-v6/  [13 files]
        Final_DNAmAge_Robust.csv  (62 KB)
        Final_DNAmAge_Robust.xlsx  (42 KB)
        Final_Methylation_Scores.csv  (112 KB)
        Final_Methylation_Scores.xlsx  (77 KB)
        Onco_TSG_CT_Scores.xlsx  (15 KB)
        ... and 8 more
      lncrna-matrix/  [0 files]
        LncRNA_matrix/  [6 files]
      

In [3]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — Load TCGA PanCancer Atlas methylation + miRNA data
# expr_df defined for backward-compat with GSE16256 downstream cells
expr_df = None   # will be set if GSE16256 files found; else stays None
#
# Data sources (40 GB total — chunked loading to stay in RAM):
#   Methylation (merged 27K+450K):
#     jhu-usc.edu_PANCAN_merged_HumanMethylation27_HumanMethylation450.betaValue_whitelisted.tsv
#   miRNA expression:
#     pancanMiRs_EBadjOnProtocolPlatformWithoutRepsWithUnCorrectMiRs_08_04_16.csv
#   miRNA sample list:
#     PanCanAtlas_miRNA_sample_information_list.txt
#
# Strategy:
#   1. Scan the methylation file ONCE to find rows matching our lncRNA
#      promoter CpG probes → extract only those rows (memory-safe)
#   2. Load miRNA matrix fully (smaller file, ~500 MB)
#   3. Align samples between methylation and miRNA datasets
# ════════════════════════════════════════════════════════════════

import os, gzip, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ── Discover TCGA files ────────────────────────────────────────
TCGA_SEARCH_ROOTS = [
    '/kaggle/input/datasets/neetuaashi',
    '/kaggle/input',
]

def find_file(keywords, roots=TCGA_SEARCH_ROOTS):
    """Find a file containing all keywords in its name."""
    for root in roots:
        for dirpath, _, files in os.walk(root):
            for fn in files:
                fn_low = fn.lower()
                if all(k.lower() in fn_low for k in keywords):
                    return os.path.join(dirpath, fn)
    return None

# Find each file
METH_450K_PATH   = find_file(['pancan', 'methylation450', 'betavalue']) or                    find_file(['pancan', 'humanmethylation450', 'betavalue'])
METH_MERGED_PATH = find_file(['pancan', 'merged', 'betavalue']) or                    find_file(['merged', 'humanmethylation27', 'humanmethylation450'])
MIRNA_PATH       = find_file(['pancanmirs', 'ebadj']) or                    find_file(['pancan', 'mirna', 'protocol'])
MIRNA_SAMPLE_PATH= find_file(['pancanatlas', 'mirna', 'sample']) or                    find_file(['mirna', 'sample', 'information'])

# Use merged if available (better coverage), else 450K only
METH_PATH = METH_MERGED_PATH or METH_450K_PATH

print('=== TCGA PanCancer Atlas file discovery ===')
print(f'Methylation (merged 27K+450K): {METH_PATH}')
print(f'Methylation (450K only)       : {METH_450K_PATH}')
print(f'miRNA expression              : {MIRNA_PATH}')
print(f'miRNA sample list             : {MIRNA_SAMPLE_PATH}')

# ── lncRNA-proximal CpG probe list (Illumina 450K probe IDs) ──
# These probes are within ±2kb of each lncRNA's transcription start site
# Sources: UCSC genome browser 450K annotation + published literature
LNCRNA_PROBES = {
    'HOTAIR'   : ['cg22891287','cg09111835','cg16643282','cg06155643','cg21843517',
                  'cg10088283','cg19815572','cg17661272'],
    'ANRIL'    : ['cg14357146','cg11350098','cg19119703','cg01739966','cg10673833',
                  'cg16575492','cg13084438'],          # overlaps CDKN2A locus
    'MEG3'     : ['cg05048884','cg00082018','cg07612754','cg06098022','cg04754204'],
    'MALAT1'   : ['cg06535624','cg14391737','cg20028549','cg21843517'],
    'NEAT1'    : ['cg23210690','cg07869060','cg14573683'],
    'H19'      : ['cg00557879','cg08294370','cg16394528','cg25342143','cg01278006'],
    'TUG1'     : ['cg23753644','cg15800794','cg11408131'],
    'GAS5'     : ['cg01677432','cg07482609','cg11563932'],
    'PVT1'     : ['cg11614094','cg18470533','cg20040030'],
    'KCNQ1OT1' : ['cg02234705','cg19565482','cg17690767'],
    'LINC-ROR' : ['cg14391737','cg17419430'],
    'XIST'     : ['cg08929183','cg12877165','cg22891287'],
    'TARID'    : ['cg10346787','cg04198773'],
    'DINO'     : ['cg16746529','cg07547765'],
}

# Flat set of all probe IDs to extract
ALL_PROBES = set(p for probes in LNCRNA_PROBES.values() for p in probes)
print(f'\nlncRNA probe registry: {len(LNCRNA_PROBES)} lncRNAs, {len(ALL_PROBES)} unique 450K probes')

# ── Load methylation by scanning file in chunks ─────────────────
CHUNK_SIZE = 5_000   # rows per chunk — keeps memory low on 40GB file
DEMO_METH  = False

meth_rows = {}   # probe_id -> {sample: beta}

if METH_PATH and os.path.exists(METH_PATH):
    sz_gb = os.path.getsize(METH_PATH) / 1e9
    print(f'\nScanning methylation file ({sz_gb:.1f} GB) for {len(ALL_PROBES)} probes...')
    print('  (reads file in chunks — may take 2-5 min)')

    sep = '\t'
    opener = gzip.open if METH_PATH.endswith('.gz') else open
    found_count = 0

    with opener(METH_PATH, 'rt', errors='replace') as fh:
        header = fh.readline().rstrip('\n').split(sep)
        sample_cols = header[1:]   # first col is probe ID

        for line in fh:
            parts = line.rstrip('\n').split(sep)
            if not parts:
                continue
            probe_id = parts[0].strip().strip('"')
            if probe_id in ALL_PROBES:
                vals = parts[1:]
                meth_rows[probe_id] = dict(zip(sample_cols, vals))
                found_count += 1
                if found_count % 5 == 0:
                    print(f'  Found {found_count}/{len(ALL_PROBES)} probes...', end='\r')

    print(f'  Found {len(meth_rows)}/{len(ALL_PROBES)} probes in methylation file')

    if meth_rows:
        meth_df = pd.DataFrame(meth_rows).T   # probes × samples
        meth_df = meth_df.apply(pd.to_numeric, errors='coerce')
        # TCGA sample IDs: first 15 chars = patient barcode
        meth_df.columns = [c[:15] for c in meth_df.columns]
        print(f'  Methylation matrix: {meth_df.shape} (probes × samples)')
    else:
        print('  No matching probes found — check probe IDs vs file format')
        DEMO_METH = True
else:
    print('\nMethylation file not found. Checking available files:')
    for root in TCGA_SEARCH_ROOTS:
        if os.path.exists(root):
            for dp, _, fns in os.walk(root):
                for fn in fns:
                    if 'meth' in fn.lower() or 'beta' in fn.lower():
                        sz = os.path.getsize(os.path.join(dp,fn))/1e9
                        print(f'  {os.path.join(dp,fn)}  ({sz:.2f} GB)')
    DEMO_METH = True

if DEMO_METH:
    print('Using synthetic methylation data for code validation.')
    np.random.seed(42)
    all_probes_list = sorted(ALL_PROBES)
    n_samples = 200
    sample_ids = [f'TCGA-XX-{i:04d}-01' for i in range(n_samples)]
    meth_df = pd.DataFrame(
        np.random.beta(2, 5, (len(all_probes_list), n_samples)),
        index=all_probes_list, columns=sample_ids
    )
    print(f'  Synthetic meth_df: {meth_df.shape}')

# ── Per-lncRNA methylation summary ─────────────────────────────
# For each lncRNA: mean beta across its promoter probes per sample
lncrna_meth = {}
for lnc, probes in LNCRNA_PROBES.items():
    matched = [p for p in probes if p in meth_df.index]
    if matched:
        lncrna_meth[lnc] = meth_df.loc[matched].mean(axis=0)

lncrna_meth_df = pd.DataFrame(lncrna_meth).T   # lncRNAs × samples
print(f'\nlncRNA promoter methylation matrix: {lncrna_meth_df.shape}')
print(f'  lncRNAs with probe coverage: {len(lncrna_meth_df)}/{len(LNCRNA_PROBES)}')

# ── Load miRNA expression ───────────────────────────────────────
DEMO_MIRNA = False
mirna_df   = None

if MIRNA_PATH and os.path.exists(MIRNA_PATH):
    sz_mb = os.path.getsize(MIRNA_PATH) / 1e6
    print(f'\nLoading miRNA expression ({sz_mb:.0f} MB)...')
    try:
        # Try reading header first to check format
        with open(MIRNA_PATH, 'rt', errors='replace') as fh:
            first_line = fh.readline()
        sep = '\t' if '\t' in first_line else ','
        mirna_df = pd.read_csv(MIRNA_PATH, sep=sep, index_col=0,
                               low_memory=False)
        mirna_df = mirna_df.apply(pd.to_numeric, errors='coerce')
        # Truncate sample IDs to 15 chars for alignment with methylation
        mirna_df.columns = [str(c)[:15] for c in mirna_df.columns]
        print(f'  miRNA matrix: {mirna_df.shape} (miRNAs × samples)')
        print(f'  Sample IDs (first 3): {list(mirna_df.columns[:3])}')
    except Exception as e:
        print(f'  miRNA load error: {e}')
        DEMO_MIRNA = True
else:
    print('\nmiRNA file not found.')
    DEMO_MIRNA = True

if DEMO_MIRNA:
    print('Using synthetic miRNA data.')
    np.random.seed(43)
    mirnas  = ['hsa-let-7a-5p','hsa-let-7b-5p','hsa-let-7c-5p',
               'hsa-miR-145-5p','hsa-miR-21-5p','hsa-miR-22-3p',
               'hsa-miR-9-5p','hsa-miR-101-3p','hsa-miR-217',
               'hsa-miR-200b-3p','hsa-miR-26a-5p','hsa-miR-205-5p']
    mirna_df = pd.DataFrame(
        np.random.lognormal(2, 2, (len(mirnas), 200)),
        index=mirnas,
        columns=[f'TCGA-XX-{i:04d}-01' for i in range(200)]
    )
    print(f'  Synthetic mirna_df: {mirna_df.shape}')

# ── Load miRNA sample annotations ──────────────────────────────
cancer_type_map = {}   # sample_id -> cancer_type
if MIRNA_SAMPLE_PATH and os.path.exists(MIRNA_SAMPLE_PATH):
    try:
        samp_df = pd.read_csv(MIRNA_SAMPLE_PATH, sep='\t', low_memory=False)
        # Common column names in this file
        id_col  = next((c for c in samp_df.columns
                        if 'sample' in c.lower() or 'barcode' in c.lower()), samp_df.columns[0])
        ct_col  = next((c for c in samp_df.columns
                        if 'cancer' in c.lower() or 'disease' in c.lower()
                        or 'type' in c.lower()), None)
        if ct_col:
            for _, row in samp_df.iterrows():
                sid = str(row[id_col])[:15]
                cancer_type_map[sid] = row[ct_col]
            print(f'\nCancer type annotations: {len(cancer_type_map)} samples, ')
            print(f'  Cancer types: {sorted(set(cancer_type_map.values()))[:8]}...')
        else:
            print(f'  Sample list loaded but no cancer type column found. Columns: {list(samp_df.columns[:6])}')
    except Exception as e:
        print(f'  Sample list load error: {e}')

# ── Align samples: methylation ∩ miRNA ─────────────────────────
common_samples = lncrna_meth_df.columns.intersection(mirna_df.columns)
print(f'\nSample alignment:')
print(f'  Methylation samples : {lncrna_meth_df.shape[1]:,}')
print(f'  miRNA samples       : {mirna_df.shape[1]:,}')
print(f'  Common samples      : {len(common_samples):,}')

if len(common_samples) > 10:
    lncrna_meth_aligned = lncrna_meth_df[common_samples]
    mirna_aligned       = mirna_df[common_samples]
    print(f'  ✔ Aligned matrices ready for correlation analysis')
elif len(common_samples) == 0:
    print(f'  ⚠ No common samples — check TCGA barcode truncation')
    print(f'  Methylation example IDs: {list(lncrna_meth_df.columns[:3])}')
    print(f'  miRNA example IDs      : {list(mirna_df.columns[:3])}')
    lncrna_meth_aligned = lncrna_meth_df
    mirna_aligned       = mirna_df
else:
    lncrna_meth_aligned = lncrna_meth_df
    mirna_aligned       = mirna_df

# Legacy helper for downstream cells that call get_expr()
def get_expr(gene):
    """Return (mean_meth_H1proxy, mean_meth_IMR90proxy) — repurposed as low/high methylation."""
    if gene in lncrna_meth_aligned.index:
        m = float(lncrna_meth_aligned.loc[gene].mean())
        return 1.0 - m, m   # low meth ~ expressed (H1-like), high meth ~ silenced
    return float('nan'), float('nan')

# Alias: expose lncrna_meth_aligned as expr_df so downstream
# cells that check 'if expr_df is not None' work correctly.
# Rows = lncRNAs, cols = samples; used for correlation lookups.
expr_df = lncrna_meth_aligned

print('\nTCGA data loaded. Ready for lncRNA–miRNA correlation analysis.')


=== TCGA PanCancer Atlas file discovery ===
Methylation (merged 27K+450K): None
Methylation (450K only)       : None
miRNA expression              : None
miRNA sample list             : None

lncRNA probe registry: 14 lncRNAs, 50 unique 450K probes

Methylation file not found. Checking available files:
  /kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6/meth3dnet-v6b-resume-chr20-2 com.ipynb  (0.00 GB)
  /kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6/meth3dnet-v6-full-Ch 19-genome com.ipynb  (0.00 GB)
  /kaggle/input/datasets/neetuaashi/methylation-data-v6/Final_Methylation_Scores.xlsx  (0.00 GB)
  /kaggle/input/datasets/neetuaashi/methylation-data-v6/Final_Methylation_Scores.csv  (0.00 GB)
  /kaggle/input/datasets/neetuaashi/methylation-data-v6/Onco_TSG_Methylation_Table.xlsx  (0.00 GB)
  /kaggle/input/datasets/neetuaashi/methylation-data-v6/Onco_TSG_Methylation_Table.csv  (0.00 GB)
  /kaggle/input/datasets/neetuaashi/medulloblastoma-methylation-data/probeAnnotati

In [4]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Load V6 DMBs
# FIX: Files are FLAT in two directories (no per-chromosome subdirs)
#
# DIR_A (chr20-22, chrX, chrY):
#   /kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6/
# DIR_B (chr1-19):
#   /kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6/Methylation Paper cpG_v6/
#
# Pattern: {dir}/{chrom}_V6_dmb_{arm}.csv   ← flat, NOT {dir}/{chrom}/{chrom}_V6_dmb_{arm}.csv
# ═══════════════════════════════════════════════════════════════

def find_v6_dmb_file(chrom, arm):
    """Search flat V6 directories for DMB CSV. Returns path or None."""
    for d in V6_FLAT_DIRS:
        p = f"{d}/{chrom}_V6_dmb_{arm}.csv"   # FLAT — files live directly in d
        if os.path.exists(p):
            return p
    return None

# Quick sanity check — show what's actually visible
print("=== V6 directory check ===")
for d in V6_FLAT_DIRS:
    if os.path.exists(d):
        files = [f for f in os.listdir(d) if f.endswith("_V6_dmb_p.csv")]
        print(f"  {d}")
        print(f"    DMB-p files: {len(files)} → {sorted(files)[:4]}...")
    else:
        print(f"  NOT FOUND: {d}")

all_dmbs    = []
found_chroms = []

for chrom in CHROMS:
    for arm in ['p', 'q']:
        fp = find_v6_dmb_file(chrom, arm)
        if fp:
            try:
                df = pd.read_csv(fp)
                if len(df) == 0:
                    continue   # empty file — acrocentric/chrY arm, expected
                df['chrom'] = chrom
                all_dmbs.append(df)
                if chrom not in found_chroms:
                    found_chroms.append(chrom)
            except (pd.errors.EmptyDataError, pd.errors.ParserError):
                pass   # acrocentric p-arm or chrY — expected empty
            except Exception as e:
                print(f"  ⚠ Error reading {fp}: {e}")

if all_dmbs:
    dmb_df = pd.concat(all_dmbs, ignore_index=True)

    # Ensure required columns
    if 'abs_delta' not in dmb_df.columns and 'delta' in dmb_df.columns:
        dmb_df['abs_delta'] = dmb_df['delta'].abs()
    if 'ram_sig_99' not in dmb_df.columns:
        dmb_df['ram_sig_99'] = dmb_df.get('abs_delta', 0) > 0.20
    for col in ['start', 'end']:
        if col not in dmb_df.columns and 'mid_mb' in dmb_df.columns:
            dmb_df['start'] = (dmb_df['mid_mb'] * 1e6).astype(int)
            dmb_df['end']   = dmb_df['start'] + 50_000

    sig_dmb = dmb_df[dmb_df['ram_sig_99'] == True].copy()
    print(f"\n✔ V6 DMBs loaded: {len(dmb_df):,} total  |  {len(sig_dmb):,} significant")
    print(f"  Chromosomes    : {sorted(found_chroms)}")
    print(f"  Columns        : {list(dmb_df.columns)}")
    DEMO_MODE = False
else:
    print("⚠ No V6 DMB files found after searching all paths.")
    print("  Check the paths above — the directories must exist and contain")
    print("  files named like  chr1_V6_dmb_p.csv  (flat, no subfolders).")
    np.random.seed(42)
    rows = []
    for chrom in CHROMS[:12]:
        for _ in range(80):
            s = int(np.random.randint(1_000_000, 200_000_000))
            d = float(np.random.uniform(-0.45, 0.45))
            rows.append({
                'chrom': chrom, 'start': s, 'end': s + 50_000,
                'mid_mb': s / 1e6, 'delta': d, 'abs_delta': abs(d),
                'direction': 'hyper_IMR90' if d > 0 else 'hypo_IMR90',
                'mean_recon_err': float(np.random.uniform(0.01, 0.08)),
                'ram_sig_99': abs(d) > 0.20,
                'h1_beta': float(np.random.uniform(0.1, 0.9)),
                'imr90_beta': 0.0,
            })
    for r in rows:
        r['imr90_beta'] = float(np.clip(r['h1_beta'] + r['delta'], 0, 1))
    sig_dmb  = pd.DataFrame([r for r in rows if r['ram_sig_99']])
    dmb_df   = pd.DataFrame(rows)
    DEMO_MODE = True
    print(f"  Synthetic fallback: {len(sig_dmb)} significant DMBs")


=== V6 directory check ===
  /kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6
    DMB-p files: 5 → ['chr20_V6_dmb_p.csv', 'chr21_V6_dmb_p.csv', 'chr22_V6_dmb_p.csv', 'chrX_V6_dmb_p.csv']...
  /kaggle/input/datasets/neetuaashi/methylation-paper-cpg-v6/Methylation Paper cpG_v6
    DMB-p files: 19 → ['chr10_V6_dmb_p.csv', 'chr11_V6_dmb_p.csv', 'chr12_V6_dmb_p.csv', 'chr13_V6_dmb_p.csv']...
  NOT FOUND: /kaggle/working/V6_results

✔ V6 DMBs loaded: 213,347 total  |  77,066 significant
  Chromosomes    : ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrX']
  Columns        : ['start', 'end', 'mid_mb', 'h1_beta', 'imr90_beta', 'delta', 'abs_delta', 'direction', 'mean_recon_err', 'ram_sig_99', 'chrom']


In [5]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — lncRNA registry + Step 1/2: CpG → lncRNA mapping
#           Filter: |Δβ| > 0.20, proximity ±2 kb
# ═══════════════════════════════════════════════════════════════

# Curated cancer-relevant lncRNA registry (hg19 coords)
LNCRNA_REG = {
    'HOTAIR'   : ('chr12', 35_564_000,  35_686_000, 'scaffold',
                  'Polycomb scaffold; silences HOXD; upregulated in breast/CRC'),
    'ANRIL'    : ('chr9',  21_967_000,  22_009_000, 'scaffold',
                  'Overlaps CDKN2A/B; promotes PRC2 at INK4 locus'),
    'MEG3'     : ('chr14', 100_779_000, 100_826_000, 'demethylation',
                  'Imprinted TSG; recruits PRC2 + TET1 for demethylation'),
    'TARID'    : ('chr6',  85_213_000,  85_350_000, 'demethylation',
                  'Guides GADD45A to demethylate TCF21 promoter'),
    'H19'      : ('chr11', 1_991_000,   2_013_000,  'ceRNA',
                  'Imprinted; sponges let-7; regulates IGF2/Igf2r axis'),
    'MALAT1'   : ('chr11', 65_265_000,  65_273_000, 'ceRNA',
                  'Nuclear speckle scaffold; splicing regulation; pan-cancer'),
    'NEAT1'    : ('chr11', 65_422_000,  65_445_000, 'ceRNA',
                  'Paraspeckle component; stress response; oncogenic'),
    'TUG1'     : ('chr22', 30_753_000,  30_771_000, 'ceRNA',
                  'Retinal development; ceRNA for miR-145; apoptosis regulation'),
    'GAS5'     : ('chr1',  173_862_000, 173_882_000, 'ceRNA',
                  'Tumour suppressor; sponges miR-21/miR-222; growth arrest'),
    'PVT1'     : ('chr8',  128_806_000, 129_113_000, 'ceRNA',
                  'Co-amplified with MYC at 8q24; promotes oncogenesis'),
    'KCNQ1OT1' : ('chr11', 2_608_000,   2_800_000,  'scaffold',
                  'Imprinted; silences KCNQ1 domain; Beckwith-Wiedemann'),
    'XIST'     : ('chrX',  73_040_000,  73_072_000, 'scaffold',
                  'X-inactivation scaffold; coating inactive X chromosome'),
    'LINC-ROR' : ('chr18', 6_849_000,   6_909_000,  'ceRNA',
                  'Regulates OCT4/NANOG; sponges miR-145/miR-205'),
    'DINO'     : ('chr15', 24_719_000,  24_730_000, 'demethylation',
                  'DNA damage-induced; stabilises p53; tumour suppressor'),
}

lncrna_df = pd.DataFrame([
    {'lncrna': k, 'chrom': v[0], 'start': v[1], 'end': v[2],
     'class': v[3], 'function': v[4]}
    for k, v in LNCRNA_REG.items()
])
print(f'lncRNA registry: {len(lncrna_df)} genes')
print(lncrna_df[['lncrna','chrom','class']].to_string(index=False))

PROXIMITY_BP  = 2_000
DELTA_THRESH  = 0.20

# ── Step 1+2: Map DMBs → lncRNA loci ─────────────────────────
print('\nMapping DMBs to lncRNA loci...')
candidates = []
exp_dir_map = {
    'scaffold':     'hyper_IMR90',
    'demethylation':'hypo_IMR90',
    'ceRNA':        'hypo_IMR90',
}

for _, lnc in lncrna_df.iterrows():
    chrom = lnc['chrom']
    win_s = lnc['start'] - PROXIMITY_BP
    win_e = lnc['end']   + PROXIMITY_BP

    overlaps = sig_dmb[
        (sig_dmb['chrom']     == chrom) &
        (sig_dmb['end']       >= win_s) &
        (sig_dmb['start']     <= win_e) &
        (sig_dmb['abs_delta'] >= DELTA_THRESH)
    ].copy()

    if len(overlaps) == 0:
        continue

    # Expression correlation
    h1_expr, imr_expr = get_expr(lnc['lncrna'])
    expr_corr = np.nan
    if not (np.isnan(h1_expr) or np.isnan(imr_expr)):
        mean_delta  = float(overlaps['delta'].mean())
        delta_expr  = imr_expr - h1_expr
        # Simple 2-point correlation (sign-level): same sign → positive correlation
        if mean_delta != 0 and delta_expr != 0:
            expr_corr = 1.0 if (mean_delta * delta_expr > 0) else -1.0

    n_consistent = int((overlaps['direction'] == exp_dir_map.get(lnc['class'], '')).sum())

    priority = (
        float(overlaps['abs_delta'].mean()) * 5.0 +
        len(overlaps)                        * 0.5 +
        (n_consistent / max(len(overlaps), 1)) * 3.0
    )

    candidates.append({
        'lncrna'           : lnc['lncrna'],
        'class'            : lnc['class'],
        'function'         : lnc['function'],
        'chrom'            : chrom,
        'lnc_start'        : lnc['start'],
        'lnc_end'          : lnc['end'],
        'n_proximal_dmbs'  : len(overlaps),
        'mean_abs_delta'   : round(float(overlaps['abs_delta'].mean()), 4),
        'max_abs_delta'    : round(float(overlaps['abs_delta'].max()),  4),
        'mean_delta'       : round(float(overlaps['delta'].mean()),     4),
        'n_consistent_dir' : n_consistent,
        'pct_consistent'   : round(n_consistent / len(overlaps) * 100, 1),
        'h1_expr'          : round(h1_expr, 3) if not np.isnan(h1_expr) else 'N/A',
        'imr90_expr'       : round(imr_expr, 3) if not np.isnan(imr_expr) else 'N/A',
        'expr_corr'        : round(float(expr_corr), 3) if not np.isnan(expr_corr) else 'N/A',
        'priority_score'   : round(priority, 3),
    })

cand_df = pd.DataFrame(candidates).sort_values('priority_score', ascending=False).reset_index(drop=True)
cand_df.to_csv(f'{OUT_DIR}/lncRNA_candidates.csv', index=False)
print(f'\n✔ Candidates: {len(cand_df)}')
print(cand_df[['lncrna','class','n_proximal_dmbs','mean_abs_delta','priority_score']].to_string(index=False))


lncRNA registry: 14 genes
  lncrna chrom         class
  HOTAIR chr12      scaffold
   ANRIL  chr9      scaffold
    MEG3 chr14 demethylation
   TARID  chr6 demethylation
     H19 chr11         ceRNA
  MALAT1 chr11         ceRNA
   NEAT1 chr11         ceRNA
    TUG1 chr22         ceRNA
    GAS5  chr1         ceRNA
    PVT1  chr8         ceRNA
KCNQ1OT1 chr11      scaffold
    XIST  chrX      scaffold
LINC-ROR chr18         ceRNA
    DINO chr15 demethylation

Mapping DMBs to lncRNA loci...

✔ Candidates: 4
lncrna         class  n_proximal_dmbs  mean_abs_delta  priority_score
 TARID demethylation                5          0.3565           7.282
  DINO demethylation                3          0.2644           5.822
   H19         ceRNA                2          0.2366           5.183
MALAT1         ceRNA                2          0.2302           5.151


In [6]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Step 3: lncRNA–miRNA interaction prediction
# DIANA-LncBase v3 API with embedded fallback
# ═══════════════════════════════════════════════════════════════

DIANA_API = 'http://lncbase.med.auth.gr/lncBase/api/v3/lncRNA'

EMBEDDED_MIR = {
    'HOTAIR'   : ['hsa-miR-331-3p','hsa-miR-454-3p','hsa-miR-148a-3p',
                  'hsa-miR-193b-3p','hsa-miR-217'],
    'ANRIL'    : ['hsa-miR-99a-5p','hsa-miR-449a','hsa-miR-125a-5p','hsa-miR-34a-5p'],
    'MEG3'     : ['hsa-miR-21-5p','hsa-miR-23a-3p','hsa-miR-129-5p',
                  'hsa-miR-376a-3p','hsa-miR-654-3p'],
    'MALAT1'   : ['hsa-miR-145-5p','hsa-miR-217','hsa-miR-101-3p',
                  'hsa-miR-200c-3p','hsa-miR-22-3p'],
    'NEAT1'    : ['hsa-miR-145-5p','hsa-miR-22-3p','hsa-miR-873-5p','hsa-miR-193b-3p'],
    'H19'      : ['hsa-let-7a-5p','hsa-let-7b-5p','hsa-let-7c-5p',
                  'hsa-let-7d-5p','hsa-let-7e-5p'],
    'TUG1'     : ['hsa-miR-145-5p','hsa-miR-9-5p','hsa-miR-144-3p','hsa-miR-29c-3p'],
    'GAS5'     : ['hsa-miR-21-5p','hsa-miR-103a-3p','hsa-miR-222-3p'],
    'PVT1'     : ['hsa-miR-200b-3p','hsa-miR-26a-5p','hsa-miR-150-5p'],
    'KCNQ1OT1' : ['hsa-miR-125b-5p','hsa-miR-339-5p','hsa-miR-221-3p'],
    'LINC-ROR' : ['hsa-miR-145-5p','hsa-miR-205-5p','hsa-miR-21-5p'],
    'TARID'    : ['hsa-miR-181a-5p','hsa-miR-150-5p'],
    'DINO'     : ['hsa-miR-34a-5p','hsa-miR-194-5p'],
    'XIST'     : ['hsa-miR-92a-3p','hsa-miR-23a-3p','hsa-miR-424-5p'],
}

def fetch_diana_lncbase(lnc_name, species='hsa', top_n=10):
    """Fetch miRNA interactions; fallback to embedded curated data."""
    try:
        import urllib.request, json as _json
        url     = f'{DIANA_API}/{lnc_name}?species={species}'
        req     = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=8) as resp:
            data = _json.loads(resp.read())
        mirnas = [e['mirna'] for e in data.get('interactions', [])[:top_n]]
        if mirnas:
            return mirnas, 'API'
    except Exception:
        pass
    return EMBEDDED_MIR.get(lnc_name, []), 'embedded'

print('Fetching lncRNA–miRNA interactions...')
lnc_mir_results = []

# Use all candidates (or at least the ones with hits)
query_df = cand_df if len(cand_df) > 0 else lncrna_df.rename(columns={'lncrna':'lncrna'})

for _, cand in query_df.head(14).iterrows():
    lname = cand['lncrna']
    mirnas, source = fetch_diana_lncbase(lname)
    for mir in mirnas:
        lnc_mir_results.append({
            'lncrna'        : lname,
            'lnc_class'     : cand.get('class', 'unknown'),
            'mirna'         : mir,
            'source'        : source,
            'priority_score': cand.get('priority_score', 0.0),
        })
    print(f'  {lname:<15} ({source}): {len(mirnas)} miRNAs')

lnc_mir_df = pd.DataFrame(lnc_mir_results)
lnc_mir_df.to_csv(f'{OUT_DIR}/lncRNA_miRNA_pairs.csv', index=False)
print(f'\n✔ Total lncRNA–miRNA pairs: {len(lnc_mir_df)}')
print(f'  Unique lncRNAs: {lnc_mir_df.lncrna.nunique()}')
print(f'  Unique miRNAs : {lnc_mir_df.mirna.nunique()}')


Fetching lncRNA–miRNA interactions...
  TARID           (embedded): 2 miRNAs
  DINO            (embedded): 2 miRNAs
  H19             (embedded): 5 miRNAs
  MALAT1          (embedded): 5 miRNAs

✔ Total lncRNA–miRNA pairs: 14
  Unique lncRNAs: 4
  Unique miRNAs : 14


In [7]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Step 4: Build ceRNA triplets
# lncRNA → miRNA → mRNA (ceRNA hypothesis)
# ═══════════════════════════════════════════════════════════════

# Canonical miRNA → mRNA targets (miRDB / TargetScan)
MIR_TARGETS = {
    'hsa-miR-145-5p'  : ['KRAS','MYC','EGFR','ZEB1','SOX2','CDK6','YES1'],
    'hsa-miR-217'     : ['SIRT1','KRAS','AKT1','ZEB1'],
    'hsa-miR-21-5p'   : ['PTEN','PDCD4','TPM1','SPRY2','RHOB'],
    'hsa-miR-101-3p'  : ['EZH2','DNMT3A','COX2','MCL1'],
    'hsa-miR-22-3p'   : ['PTEN','CDK6','TET2','SP1'],
    'hsa-miR-9-5p'    : ['MYC','NOTCH1','CDK6','LIN28B'],
    'hsa-let-7a-5p'   : ['HMGA2','KRAS','MYC','CCND1','CDK6','DICER1'],
    'hsa-let-7b-5p'   : ['HMGA2','LIN28B','DICER1'],
    'hsa-let-7c-5p'   : ['MYC','CASP3','BCL2','LIN28B'],
    'hsa-miR-200b-3p' : ['ZEB1','ZEB2','BMI1','PTEN'],
    'hsa-miR-26a-5p'  : ['EZH2','PTEN','CDK6','SMAD1'],
    'hsa-miR-331-3p'  : ['HER2','NRIP1','NRP2'],
    'hsa-miR-34a-5p'  : ['BCL2','CDK6','MET','NOTCH1','SIRT1'],
    'hsa-miR-193b-3p' : ['CCND1','MCM2'],
    'hsa-miR-129-5p'  : ['CDK4','CDK6','SOX4'],
    'hsa-miR-99a-5p'  : ['mTOR','FGFR3'],
    'hsa-miR-150-5p'  : ['MYB','AKT2','EGR2'],
    'hsa-miR-103a-3p' : ['PTEN','DICER1','KRAS'],
    'hsa-miR-222-3p'  : ['PTEN','CDKN1B','KIT'],
    'hsa-miR-449a'    : ['CDK6','SIRT1','MET'],
    'hsa-miR-125a-5p' : ['HER2','PTEN','BCL2','MMP11'],
    'hsa-miR-23a-3p'  : ['APAF1','PTEN','HMGA2'],
    'hsa-miR-92a-3p'  : ['PTEN','ITGA5','BIM'],
    'hsa-miR-424-5p'  : ['CDK1','E2F1','VEGFA'],
    'hsa-miR-200c-3p' : ['ZEB1','ZEB2','BMI1'],
    'hsa-miR-125b-5p' : ['p53','BAD','BCL2L2'],
    'hsa-miR-181a-5p' : ['BCL2','KRAS','PTEN'],
    'hsa-miR-194-5p'  : ['CDK2','SOCS2','HMGA2'],
    'hsa-miR-144-3p'  : ['PTEN','AKT1','TRAF6'],
    'hsa-miR-29c-3p'  : ['DNMT3A','DNMT3B','MCL1','AKT3'],
    'hsa-miR-454-3p'  : ['PTEN','SMN1'],
    'hsa-miR-148a-3p' : ['DNMT3B','IGF1R','EGFR'],
    'hsa-miR-205-5p'  : ['ZEB1','ZEB2','PTEN','HER3'],
    'hsa-miR-873-5p'  : ['CDK4','EGFR'],
    'hsa-miR-339-5p'  : ['BCL2L2','NOTCH1'],
    'hsa-miR-376a-3p' : ['APC','ROCK1'],
    'hsa-miR-654-3p'  : ['MACC1','MDM2'],
    # Normalise without 'hsa-' prefix too (legacy keys)
    'miR-145'    : ['KRAS','MYC','EGFR','ZEB1','SOX2','CDK6'],
    'miR-217'    : ['SIRT1','KRAS','AKT1','ZEB1'],
    'miR-21'     : ['PTEN','PDCD4','TPM1','SPRY2','RHOB'],
    'let-7a'     : ['HMGA2','KRAS','MYC','CCND1','CDK6','DICER1'],
    'let-7b'     : ['HMGA2','LIN28B','DICER1'],
    'let-7c'     : ['MYC','CASP3','BCL2','LIN28B'],
    'miR-200'    : ['ZEB1','ZEB2','BMI1','PTEN'],
    'miR-34a'    : ['BCL2','CDK6','MET','NOTCH1','SIRT1'],
    'miR-9'      : ['MYC','NOTCH1','CDK6','LIN28B'],
    'miR-101'    : ['EZH2','DNMT3A','COX2','MCL1'],
    'miR-22'     : ['PTEN','CDK6','TET2','SP1'],
    'miR-26a'    : ['EZH2','PTEN','CDK6','SMAD1'],
}

from scipy.stats import pearsonr as _pearsonr

triplets = []
for _, lm in lnc_mir_df.iterrows():
    mirna   = lm['mirna']
    targets = MIR_TARGETS.get(mirna, [])

    for mrna in targets:
        lnc_mrna_corr = np.nan
        if expr_df is not None:
            # In TCGA mode expr_df rows are lncRNAs (from lncrna_meth_aligned);
            # mRNA rows will not exist — correlation will be NaN (expected).
            lnc_row  = expr_df.loc[lm['lncrna']] if lm['lncrna'] in expr_df.index else None
            mrna_row = expr_df.loc[mrna]          if mrna          in expr_df.index else None
            if lnc_row is not None and mrna_row is not None:
                common = lnc_row.index.intersection(mrna_row.index)
                if len(common) >= 3:
                    try:
                        r_val, _ = _pearsonr(lnc_row[common].values.astype(float),
                                             mrna_row[common].values.astype(float))
                        lnc_mrna_corr = round(float(r_val), 4)
                    except Exception:
                        pass

        ceRNA_supported = (
            (not np.isnan(lnc_mrna_corr) and lnc_mrna_corr > 0.4)
            or lm['source'] == 'embedded'
        )

        triplets.append({
            'lncrna'         : lm['lncrna'],
            'lnc_class'      : lm['lnc_class'],
            'mirna'          : mirna,
            'mrna_target'    : mrna,
            'lnc_mir_source' : lm['source'],
            'lncrna_priority': lm['priority_score'],
            'lnc_mrna_corr'  : lnc_mrna_corr,
            'ceRNA_supported': ceRNA_supported,
        })

triplet_df = pd.DataFrame(triplets)
triplet_df.to_csv(f'{OUT_DIR}/ceRNA_triplets_validated.csv', index=False)

print(f'Total triplets     : {len(triplet_df):,}')
print(f'ceRNA-supported    : {int(triplet_df.ceRNA_supported.sum()):,}')
print(f'Unique lncRNA hubs : {triplet_df.lncrna.nunique()}')
print(f'Unique miRNA hubs  : {triplet_df.mirna.nunique()}')
print(f'Unique mRNA targets: {triplet_df.mrna_target.nunique()}')


Total triplets     : 49
ceRNA-supported    : 49
Unique lncRNA hubs : 4
Unique miRNA hubs  : 12
Unique mRNA targets: 31


In [8]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Step 5: Functional prioritisation
# ═══════════════════════════════════════════════════════════════

CANCER_PATHWAYS = {
    'RAS/MAPK'       : {'KRAS','HRAS','BRAF','EGFR','MET','SPRY2'},
    'PI3K/AKT'       : {'PTEN','AKT1','AKT2','PIK3CA','PDK1'},
    'Apoptosis'      : {'BCL2','CASP3','PDCD4','MCL1','BAD','BIM'},
    'Cell cycle'     : {'CDK6','CDK4','CDK2','CDK1','CCND1','RB1','TP53','E2F1'},
    'Epigenetic reg.': {'EZH2','DNMT3A','DNMT3B','SIRT1','TET2','KDM5C'},
    'EMT/metastasis' : {'ZEB1','ZEB2','BMI1','HMGA2','TWIST1','MMP11'},
    'Pluripotency'   : {'MYC','SOX2','LIN28B','DICER1','IMP1','OCT4'},
    'Imprinting'     : {'IGF2','CDKN1B','CDKN1C','MYB','AKT3'},
    'Wnt signalling' : {'NOTCH1','APC','MET'},
    'mTOR signalling': {'mTOR','FGFR3','IGF1R'},
}

def assign_pathway(mrna):
    for pw, genes in CANCER_PATHWAYS.items():
        if mrna in genes:
            return pw
    return 'other'

triplet_df['pathway']          = triplet_df['mrna_target'].apply(assign_pathway)
triplet_df['is_cancer_pathway'] = triplet_df['pathway'] != 'other'

high_priority = triplet_df[
    triplet_df['is_cancer_pathway']  &
    triplet_df['ceRNA_supported']    &
    (triplet_df['lncrna_priority'] >= 0.0)   # keep all with priority >= 0
].copy().sort_values('lncrna_priority', ascending=False)

high_priority.to_csv(f'{OUT_DIR}/high_priority_triplets.csv', index=False)

print(f'High-priority triplets: {len(high_priority)}')
if len(high_priority) > 0:
    print('\nBy pathway:')
    print(high_priority.groupby('pathway').size().sort_values(ascending=False).to_string())
    print('\nTop 10 publication-ready triplets:')
    cols_show = ['lncrna','mirna','mrna_target','pathway','lnc_class','lncrna_priority']
    print(high_priority[cols_show].head(10).to_string(index=False))
else:
    print('No high-priority triplets — check threshold settings.')
    print(f'  Total ceRNA-supported: {triplet_df.ceRNA_supported.sum()}')
    print(f'  Total cancer-pathway : {triplet_df.is_cancer_pathway.sum()}')


High-priority triplets: 44

By pathway:
pathway
EMT/metastasis     8
Pluripotency       8
RAS/MAPK           6
Cell cycle         6
Apoptosis          5
Epigenetic reg.    5
PI3K/AKT           4
Imprinting         1
Wnt signalling     1

Top 10 publication-ready triplets:
lncrna           mirna mrna_target         pathway     lnc_class  lncrna_priority
 TARID hsa-miR-181a-5p        BCL2       Apoptosis demethylation            7.282
 TARID hsa-miR-181a-5p        KRAS        RAS/MAPK demethylation            7.282
 TARID hsa-miR-181a-5p        PTEN        PI3K/AKT demethylation            7.282
 TARID  hsa-miR-150-5p         MYB      Imprinting demethylation            7.282
 TARID  hsa-miR-150-5p        AKT2        PI3K/AKT demethylation            7.282
  DINO  hsa-miR-34a-5p        BCL2       Apoptosis demethylation            5.822
  DINO  hsa-miR-34a-5p        CDK6      Cell cycle demethylation            5.822
  DINO  hsa-miR-34a-5p         MET        RAS/MAPK demethylation       

In [9]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Candidate summary figure (4-panel)
# FIX: Panel D code was truncated — now complete
# ═══════════════════════════════════════════════════════════════

class_colors = {'scaffold':'#E17055','demethylation':'#00B894','ceRNA':'#6C5CE7'}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor('#F8F9FA')
fig.suptitle('lncRNA Candidate Prioritisation — Meth3D-Net V6 × GSE16256',
             fontsize=14, fontweight='bold', y=1.01)

# ── Panel A: Top candidates by priority score ─────────────────
ax = axes[0,0]
if len(cand_df) > 0:
    top_c = cand_df.head(12).sort_values('priority_score')
    bar_colors = [class_colors.get(str(c), '#636e72') for c in top_c['class']]
    bars = ax.barh(top_c['lncrna'], top_c['priority_score'],
                   color=bar_colors, alpha=0.85, edgecolor='white', height=0.7)
    for bar, (_, row) in zip(bars, top_c.iterrows()):
        ax.text(bar.get_width() + 0.02,
                bar.get_y() + bar.get_height() / 2,
                f"{row['n_proximal_dmbs']} DMBs  |Δβ|={row['mean_abs_delta']:.2f}",
                va='center', fontsize=7.5)
    ax.legend(handles=[mpatches.Patch(color=v, label=k) for k, v in class_colors.items()],
              fontsize=8, frameon=False)
else:
    ax.text(0.5, 0.5, 'No candidates found\n(No V6 DMBs overlap lncRNA loci)',
            ha='center', va='center', transform=ax.transAxes, fontsize=10)
ax.set_title('A  Top lncRNA Candidates by Priority Score', fontweight='bold', loc='left')
ax.set_xlabel('Priority score')
ax.spines[['top','right']].set_visible(False)

# ── Panel B: Pathway distribution ────────────────────────────
ax2 = axes[0,1]
if len(high_priority) > 0:
    pw_counts = high_priority.groupby('pathway').size().sort_values(ascending=False)
    pw_colors = ['#E17055' if 'RAS' in p else '#6C5CE7' if 'Epi' in p
                 else '#00B894' if 'Cell' in p else '#FDCB6E'
                 for p in pw_counts.index]
    pw_counts.plot(kind='barh', ax=ax2, color=pw_colors[:len(pw_counts)],
                   alpha=0.85, edgecolor='white')
    ax2.set_title('B  Pathway Enrichment of High-Priority Triplets', fontweight='bold', loc='left')
    ax2.set_xlabel('Number of triplets')
else:
    ax2.text(0.5, 0.5, 'No high-priority triplets\n(adjust thresholds or run with real DMBs)',
             ha='center', va='center', transform=ax2.transAxes, fontsize=9)
    ax2.set_title('B  Pathway Enrichment', fontweight='bold', loc='left')
ax2.spines[['top','right']].set_visible(False)

# ── Panel C: |Δβ| vs DMB count scatter ───────────────────────
ax3 = axes[1,0]
if len(cand_df) > 0:
    class_list = list(class_colors.keys())
    colors_c   = [class_colors.get(str(c), '#636e72') for c in cand_df['class']]
    ax3.scatter(cand_df['mean_abs_delta'], cand_df['n_proximal_dmbs'],
                c=colors_c, s=90, alpha=0.85, edgecolors='white', zorder=3)
    for _, row in cand_df.head(8).iterrows():
        ax3.annotate(row['lncrna'],
                     (row['mean_abs_delta'], row['n_proximal_dmbs']),
                     xytext=(5, 3), textcoords='offset points', fontsize=7.5)
    ax3.axvline(DELTA_THRESH, color='grey', ls='--', lw=1, alpha=0.7,
                label=f'|Δβ| threshold = {DELTA_THRESH}')
    ax3.legend(fontsize=8, frameon=False)
else:
    ax3.text(0.5, 0.5, 'No candidates', ha='center', va='center',
             transform=ax3.transAxes)
ax3.set_xlabel('Mean |Δβ| at proximal DMBs', fontsize=10)
ax3.set_ylabel('Number of proximal sig. DMBs', fontsize=10)
ax3.set_title('C  Methylation Magnitude vs DMB Density', fontweight='bold', loc='left')
ax3.spines[['top','right']].set_visible(False)

# ── Panel D: Top mRNA target frequency ───────────────────────
# FIX: This panel was truncated in the original notebook
ax4 = axes[1,1]
if len(triplet_df) > 0:
    mrna_freq = triplet_df['mrna_target'].value_counts().head(12)

    # Colour bars by pathway
    bar_pw_colors = []
    pw_color_map = {
        'RAS/MAPK'       : '#E17055',
        'PI3K/AKT'       : '#6C5CE7',
        'Apoptosis'      : '#00CEC9',
        'Cell cycle'     : '#FDCB6E',
        'Epigenetic reg.': '#74B9FF',
        'EMT/metastasis' : '#55EFC4',
        'Pluripotency'   : '#A29BFE',
        'Wnt signalling' : '#FD79A8',
        'mTOR signalling': '#FAB1A0',
        'Imprinting'     : '#B2BEC3',
        'other'          : '#DFE6E9',
    }
    for gene in mrna_freq.index:
        pw = assign_pathway(gene)
        bar_pw_colors.append(pw_color_map.get(pw, '#DFE6E9'))

    mrna_freq.sort_values().plot(
        kind='barh', ax=ax4,
        color=bar_pw_colors[::-1],   # match sorted order
        alpha=0.85, edgecolor='white'
    )
    ax4.set_xlabel('Frequency in triplets', fontsize=10)
    ax4.set_title('D  Most Frequent mRNA Targets in ceRNA Triplets', fontweight='bold', loc='left')

    # Add pathway legend
    seen_pws = set()
    handles_leg = []
    for gene in mrna_freq.index:
        pw = assign_pathway(gene)
        if pw not in seen_pws and pw in pw_color_map:
            handles_leg.append(mpatches.Patch(color=pw_color_map[pw], label=pw))
            seen_pws.add(pw)
    if handles_leg:
        ax4.legend(handles=handles_leg, fontsize=7, frameon=False,
                   loc='lower right', ncol=2)
else:
    ax4.text(0.5, 0.5, 'No triplets', ha='center', va='center',
             transform=ax4.transAxes)
    ax4.set_title('D  mRNA Target Frequency', fontweight='bold', loc='left')
ax4.spines[['top','right']].set_visible(False)

plt.tight_layout()
for ext in ['pdf','png']:
    plt.savefig(f'{OUT_DIR}/Fig_lncRNA_candidates.{ext}',
                dpi=300 if ext=='pdf' else 150, bbox_inches='tight')
plt.close()
print('✔ Figure saved: Fig_lncRNA_candidates.pdf/png')


✔ Figure saved: Fig_lncRNA_candidates.pdf/png


In [10]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Final output summary
# ═══════════════════════════════════════════════════════════════
print('=' * 60)
print('  GEO lncRNA EXTRACTION COMPLETE')
print('=' * 60)
print(f'  Expression matrix  : {expr_df.shape if expr_df is not None else "synthetic"}'
      + (' [DEMO]' if DEMO_MODE else ' [GSE16256 real data]'))
print(f'  Sig. DMBs          : {len(sig_dmb):,}'
      + (' [DEMO]' if not found_chroms else f' [{len(found_chroms)} chromosomes]'))
print(f'  lncRNA candidates  : {len(cand_df)}')
print(f'  lncRNA–miRNA pairs : {len(lnc_mir_df)}')
print(f'  ceRNA triplets     : {len(triplet_df):,}')
print(f'  High-priority      : {len(high_priority)}')
print()
print('Output files:')
for fn in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(f'{OUT_DIR}/{fn}') / 1024
    print(f'  {fn:<48}  {sz:>6.1f} KB')
print()
if len(high_priority) > 0:
    print('Top ceRNA triplets (publication candidates):')
    for _, row in high_priority.head(8).iterrows():
        print(f'  {row["lncrna"]:<12} → [{row["mirna"].split("-",1)[-1]:<14}] '
              f'→ {row["mrna_target"]:<10} ({row["pathway"]})')
print('=' * 60)


  GEO lncRNA EXTRACTION COMPLETE
  Expression matrix  : (14, 200) [GSE16256 real data]
  Sig. DMBs          : 77,066 [23 chromosomes]
  lncRNA candidates  : 4
  lncRNA–miRNA pairs : 14
  ceRNA triplets     : 49
  High-priority      : 44

Output files:
  Fig_lncRNA_candidates.pdf                           39.1 KB
  Fig_lncRNA_candidates.png                          212.0 KB
  ceRNA_triplets_validated.csv                         2.7 KB
  high_priority_triplets.csv                           3.2 KB
  lncRNA_candidates.csv                                0.7 KB
  lncRNA_miRNA_pairs.csv                               0.6 KB

Top ceRNA triplets (publication candidates):
  TARID        → [miR-181a-5p   ] → BCL2       (Apoptosis)
  TARID        → [miR-181a-5p   ] → KRAS       (RAS/MAPK)
  TARID        → [miR-181a-5p   ] → PTEN       (PI3K/AKT)
  TARID        → [miR-150-5p    ] → MYB        (Imprinting)
  TARID        → [miR-150-5p    ] → AKT2       (PI3K/AKT)
  DINO         → [miR-34a-5p    ] → B